### For computing solvated entropies, you will need at least two strucures - the solute and the solvent.

In [ ]:
from ase.build import molecule
from pymatgen.io.ase import AseAtomsAdaptor

water_struct = AseAtomsAdaptor.get_structure(molecule("H2O", vacuum=10.0))

### The next step will be to convert these into my `StructureVolume` object (acts like a `Structure` but can do some volume computation stuff)

In [ ]:
from JDFTxFreeNrg.volume import StructureVolume

water_sv = StructureVolume.from_structure(water_struct)

### By default, these `StructureVolume`'s will be initialized to use monte-carlo integration. Volumes for the total structure can be retrieved by the `get_volume` method. The accuracy of this integration can be controlled with `npoints` or `grid_spacing` (which does nothing for MC integration, but also is not implemented for mesh yet)

In [ ]:
water_vol = water_sv.get_volume(npoints=100000)

### Usable values might require a bit of time to compute. We can avoid re-calculating redundant volumes by setting cache directories for our structures upon initialization

In [ ]:
from os import getcwd
from pathlib import Path

h2o_cache = Path(getcwd()) / "data" / "H2O_cache"

water_sv = StructureVolume.from_structure(water_struct, cache_parent=h2o_cache)
water_sv.clear_cache()

### Now requesting the same integration a second time will recall the first value computed.

In [5]:
from time import time

start = time()
water_vol = water_sv.get_volume(npoints=1e5)
end = time()
print(f"Initial computation time: {end - start} seconds")

start = time()
water_vol = water_sv.get_volume(npoints=1e5)
end = time()
print(f"Cached computation time: {end - start} seconds")

Initial computation time: 1.3272838592529297 seconds
Cached computation time: 5.602836608886719e-05 seconds


### Now changing the accuracy parameter will trigger a reevaluation for that level of accuracy

In [6]:
start = time()
water_vol = water_sv.get_volume(npoints=1e5)
end = time()
print(f"Computation time with n = 100000: {end - start} seconds")

start = time()
water_vol = water_sv.get_volume(npoints=1e5 + 1)
end = time()
print(f"Computation time with n = 100001: {end - start} seconds")

Computation time with n = 100000: 5.793571472167969e-05 seconds
Computation time with n = 100001: 1.3150439262390137 seconds


### Not specifying the level of accuracy will automatically grab the most expensive one generated

In [7]:
start = time()
water_vol = water_sv.get_volume()
end = time()
print(f"Computation time with default npoints: {end - start} seconds")

Computation time with default npoints: 6.103515625e-05 seconds


### We may not be interested in the total VdW volume of the structure, but that of a substructure of a structure (ie a molecule desorbed from a slab). We can specify the substructure by the atom indices of the substructure when getting our volume

In [ ]:
sub_water_vol = water_sv.get_volume(idcs=[0,1], npoints=1e5)

### We can also specify the method of integration upon initialization of the `StructureVolume` object. Options are "MC", "Mesh", and "PyVol" (case insensitive)

In [9]:
water_sv = StructureVolume.from_structure(water_struct, cache_parent=h2o_cache, method="Mesh")
acetic_acid_sv = StructureVolume.from_structure(acetic_acid_struct, cache_parent=acetic_acid_cache, method="Mesh")
water_sv.clear_cache()
acetic_acid_sv.clear_cache()

### Note for mesh integration, the "npoints" parameter (now representing number of voxels in our mesh) is not 1:1 in time cost with MC grid (where "npoints" specified the number of random points sampled). On my machine the cost of mesh integration is ~100x cheaper for a given npoints, so ive changed "npoints" to 1e7. However if you were to put in "1e5" again, the caching system would compute a new value since it only has MC values archived for 1e5 points. 

In [10]:
start = time()
water_vol = water_sv.get_volume(npoints=1e7)
end = time()